# Demo of the combination and simplification algorithm

In [1]:
import json

In [2]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

## For using the library, just import the _RuleCOSIClassifier_ class from **rulecosi** package

In [3]:
from rulecosi import RuleCOSIClassifier

The algorithm works with several type of tree ensembles and it uses the **sklearn** implementations.
- Bagging Trees
- RandomForests
- Gradient Boosting Trees (original implementation)
- XGBoost
- Light GBM
- CatBoost

In [4]:
#from catboost import CatBoostClassifier
#from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

#from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer

### Load a sample dataset and split the data

We use the Wisconsin diagnostic breast cancer dataset. There are two classes, malignant (0) and benign (1).

In [5]:
data = pd.read_csv('data/simtech_exp2_new.csv')

In [6]:
data.head()

,Material_cushion_max,Screw_volume_max,Opening_force_max,Opening_speed_max,Closing_speed_max,Mould_protection_force_max,Ejector_pressure_Actual_max,Nozzle_1_pressure_max,Injection_torque_max,Injection_force_of_screw_1_max,...,Cylinder_heating_zone_5_skew,Mould_temperature_control_unit_1_skew,Nozzle_stroke_skew,Advancement_speed_skew,Retraction_speed_skew,Temperature_of_support_housing_skew,Circumferential_speed_skew,Ejector_pressure_Nominal_skew,Injection_rotational_speed_skew,Class
0,0.899195,6.754157,2.213339,381.6942,264.3233,346.2031,6.279184,80.92746,1.650,26.26057,...,-1.324671,2.673892,-1.955708,-7.589650,7.577081,0.0,2.673892,7.490697,-7.561055,0
1,6.372508,6.754157,2.028894,486.6602,266.2318,345.6498,6.599628,80.56124,3.080,22.28584,...,0.533175,0.000000,-1.654899,6.678645,7.068555,0.0,2.610800,5.370868,-7.118657,1
2,3.677187,6.754157,2.766674,616.4362,239.5131,346.2031,9.086886,81.22502,1.595,26.43147,...,2.673892,0.000000,-1.617087,-7.615773,7.615773,0.0,2.647605,7.490697,-7.557917,1
3,5.857226,6.754157,2.213339,402.6874,263.3690,346.0187,6.157110,80.31709,1.650,20.89907,...,1.614390,0.000000,-1.649554,7.545227,7.549834,0.0,2.372336,7.288440,4.236521,1
4,0.891268,6.754157,2.213339,168.8997,271.0029,345.6498,27.901550,82.42287,0.935,25.89434,...,-0.175560,0.000000,-2.273586,0.000000,-7.615773,0.0,2.673892,-3.493064,2.544864,1


In [7]:
data.describe()

,Material_cushion_max,Screw_volume_max,Opening_force_max,Opening_speed_max,Closing_speed_max,Mould_protection_force_max,Ejector_pressure_Actual_max,Nozzle_1_pressure_max,Injection_torque_max,Injection_force_of_screw_1_max,...,Cylinder_heating_zone_5_skew,Mould_temperature_control_unit_1_skew,Nozzle_stroke_skew,Advancement_speed_skew,Retraction_speed_skew,Temperature_of_support_housing_skew,Circumferential_speed_skew,Ejector_pressure_Nominal_skew,Injection_rotational_speed_skew,Class
count,900.000000,9.000000e+02,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,...,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000,900.000000
mean,2.655541,6.754157e+00,2.375651,493.127762,259.828821,347.379875,13.344973,81.708193,4.607228,23.349280,...,0.445502,-0.008272,-1.745575,0.384764,-0.236706,-0.007938,2.703846,3.847006,-1.404734,0.318889
std,2.042856,1.777345e-15,0.446108,144.459904,11.214404,9.815998,8.063805,1.124208,9.594182,13.535253,...,1.179546,0.511907,0.172108,6.096124,6.049773,0.306783,0.183257,4.734379,5.842055,0.466305
min,0.690817,6.754157e+00,1.660004,3.816942,115.462500,336.980900,6.134221,80.118720,-0.165000,2.837001,...,-5.238997,-7.549834,-3.932557,-9.189329,-12.379430,-3.456980,2.359914,-3.633785,-7.657406,0.000000
25%,1.007914,6.754157e+00,2.028894,393.145100,256.689400,340.669800,6.241035,80.818742,1.595000,12.500383,...,-0.091260,0.000000,-1.780846,-6.882524,-7.410770,0.000000,2.642534,-2.994194,-7.277146,0.000000
50%,1.458644,6.754157e+00,2.213339,560.613400,261.460600,343.620900,8.766443,81.278430,1.980000,21.189610,...,0.285974,0.000000,-1.753160,0.000000,0.000000,0.000000,2.673892,7.288440,-3.636316,0.000000
75%,4.178879,6.754157e+00,2.582229,611.665000,266.231800,346.987000,21.464047,82.651760,2.530000,33.549610,...,0.887748,0.000000,-1.706926,7.475743,6.873600,0.000000,2.683035,7.425918,4.273490,1.000000
max,6.754157,6.754157e+00,4.795568,626.932800,278.636800,371.103200,28.817100,88.335830,70.015000,55.426490,...,7.615773,4.196492,-0.850576,12.379430,9.189329,5.238997,4.982683,12.319859,9.792966,1.000000


In [8]:
data.columns

Index(['Material_cushion_max', 'Screw_volume_max', 'Opening_force_max',
       'Opening_speed_max', 'Closing_speed_max', 'Mould_protection_force_max',
       'Ejector_pressure_Actual_max', 'Nozzle_1_pressure_max',
       'Injection_torque_max', 'Injection_force_of_screw_1_max',
       ...
       'Cylinder_heating_zone_5_skew', 'Mould_temperature_control_unit_1_skew',
       'Nozzle_stroke_skew', 'Advancement_speed_skew', 'Retraction_speed_skew',
       'Temperature_of_support_housing_skew', 'Circumferential_speed_skew',
       'Ejector_pressure_Nominal_skew', 'Injection_rotational_speed_skew',
       'Class'],
      dtype='object', length=305)

In [9]:
X = data.drop(['Class'], axis=1)
y = data['Class']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.10, random_state=1212)

### Simplifying an XGBoost classifier

We create a XGBClassifier instance. The ensemble can be fitted, or it can be just instantiated and RuleCOSI will fit the ensemble first and then simplify it.

In [11]:
ens = XGBClassifier(random_state=1212)

This is done by instanciating a **RuleCOSIClassifier** class with the desired parameters, _n\_estimator_, _tree\_max\_depth_, _conf\_threshold_ and _min\_samples_.

In [12]:
rc = RuleCOSIClassifier(base_ensemble=ens, 
                        metric='f1',n_estimators=100, tree_max_depth=3, 
                        conf_threshold=0.9, cov_threshold=0.0,
                        random_state=1212, column_names=X_train.columns)

In [13]:
%%time
rc.fit(X_train, y_train)

CPU times: user 2.42 s, sys: 14.4 ms, total: 2.43 s
Wall time: 176 ms


RuleCOSIClassifier(base_ensemble=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=None,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric=None,
                                               feature_types=None, gamma=None,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=...
       'Cylinder_heating_zone_4_skew', 'Cylinder_heating_zone_5_skew',
       'Mould_temperature_control_unit_1_skew', 'Nozzle_stroke_skew',
       'Advancement_speed_skew', 'Retraction_speed_skew',
       'Temperature_of_support_housing_skew', 'Circumferential_speed_skew',
       'Ejector_pressure_Nominal_skew', 'Injection_rotational_speed_skew'],
      dtype='object', length=304),
                   conf_threshold=0.9, n_estimators=100, random_state=1212)

## Examining the simplified rules

The rules will be stored in the _simplified\_ruleset_ \_ attribute of the RuleCOSI object. The function _print\_rules_ print the rules and its heuristics on the console. It can also return a string object or a pandas DataFrame object to be used for further analysis. Additionally, the decimal digits displayed on the heuristics values and the condition thresholds can be modified with the _heuristics\_digits_ and the _condition\_digits_ parameters.

In [14]:
#rules_dict['rules'][0]['Head of the rule'].item()

In [18]:
rc.simplified_ruleset_.compute_all_classification_performance(X_train, y_train)

In [19]:
rules_dict = rc.simplified_ruleset_.print_rules(return_object='dict',heuristics_digits=4, condition_digits=1)

In [20]:
with open('context_cards/ruleset.json', 'w', encoding='utf-8') as f:
    json.dump(rules_dict, f, ensure_ascii=False, indent=4)

In [21]:
rc.simplified_ruleset_.print_rules()


cov 	conf 	supp 	samples 		rule
0.2457	1.0000	0.2457	[0,199]		r_1: (Dosage_time_max ≥ 3.090) ˄ (Material_cushion_median < 1.023) ˄ (Mould_heating_circuit_3_std ≥ 0.141) ˄ (Screw_volume_std ≥ 1.145) → [1]
0.0481	1.0000	0.0481	[0,39]		r_2: (Dosage_time_max ≥ 3.090) ˄ (Material_cushion_median < 0.951) ˄ (Opening_force_median < -340.209) → [1]
0.0086	1.0000	0.0086	[0,7]		r_3: (Material_cushion_median < 0.951) ˄ (Maximum_injection_pressure_max < 1728.687) ˄ (Mould_heating_circuit_3_std ≥ 0.141) ˄ (Screw_volume_std ≥ 1.145) → [1]
0.0074	1.0000	0.0074	[0,6]		r_4: (Material_cushion_median < 1.023) ˄ (Opening_force_median < -358.192) ˄ (Opening_force_median ≥ -359.852) → [1]
0.0062	1.0000	0.0062	[0,5]		r_5: (Material_cushion_median < 0.899) ˄ (Maximum_injection_pressure_max < 1728.687) → [1]
0.6840	1.0000	0.6840	[554,0]		r_6: ( ) → [0]



In [14]:
rc.simplified_ruleset_.print_rules(return_object='dataframe',heuristics_digits=4, condition_digits=1)

,cov,conf,supp,samples,#,A,y
0,0.5928,1.0000,0.5928,"[0,364]",r_1,(BareNuclei < 2.5) ˄ (CellSize < 3.5) ˄ (Norma...,[1]
1,0.0244,1.0000,0.0244,"[0,15]",r_2,(BareNuclei < 4.5) ˄ (BlandChromatin < 3.5) ˄ ...,[1]
2,0.0163,1.0000,0.0163,"[0,10]",r_3,(BareNuclei < 8.5) ˄ (BlandChromatin < 3.5) ˄ ...,[1]
3,0.0033,1.0000,0.0033,"[0,2]",r_4,(CellSize < 2.5) ˄ (ClumpThickness < 3.5),[1]
4,0.0016,1.0000,0.0016,"[0,1]",r_5,(BareNuclei < 2.5) ˄ (CellSize < 3.5) ˄ (Mitos...,[1]
5,0.3616,0.9640,0.3485,"[214,8]",r_6,(),[0]


## Checking the classification performance of the simplified rule-based classifier

In [15]:
# this function is used for counting the number of rules extracted from the tree ensemble (original ruelesets)
def get_n_rules(rulesets):
    n_rules = 0
    for ruleset in rulesets:
        for rule in ruleset:
            n_rules += 1
    return n_rules

In [21]:
print(f'== Original XGBoost ensemble ==')
print(f'Number of trees: {rc.base_ensemble_.n_estimators} trees')
print(f'Number of rules: {get_n_rules(rc.original_rulesets_)} rules\n')

print(f'== Simplified rules ==')
rc.simplified_ruleset_.print_rules()
y_pred = rc.predict(X_test)
if isinstance(rc.base_ensemble, XGBClassifier):
    y_pred_ens = rc.base_ensemble_.predict(X_test, validate_features=False)
else:
    y_pred_ens = rc.base_ensemble_.predict(X_test)
print("Combinations: {}".format(rc.n_combinations_))
print("Time: {}\n".format(rc.combination_time_))
print(f'====== Classification performance of XGBoost ======')
print(classification_report(y_test, y_pred_ens,digits=4))
print(f'\n====== Classification performance of simplified rules ======')
print(classification_report(y_test, y_pred,digits=4))
print('\n')


== Original XGBoost ensemble ==
Number of trees: 100 trees
Number of rules: 499 rules

== Simplified rules ==
cov 	conf 	supp 	samples 		rule
0.5928	1.0000	0.5928	[0,364]		r_1: (BareNuclei < 2.500) ˄ (CellSize < 3.500) ˄ (NormalNucleoli < 3.500) → [1]
0.0244	1.0000	0.0244	[0,15]		r_2: (BareNuclei < 4.500) ˄ (BlandChromatin < 3.500) ˄ (CellShape < 2.500) → [1]
0.0163	1.0000	0.0163	[0,10]		r_3: (BareNuclei < 8.500) ˄ (BlandChromatin < 3.500) ˄ (ClumpThickness < 6.500) ˄ (MarginalAdhesion < 2.500) ˄ (Mitoses < 2.500) ˄ (NormalNucleoli < 8.500) → [1]
0.0033	1.0000	0.0033	[0,2]		r_4: (CellSize < 2.500) ˄ (ClumpThickness < 3.500) → [1]
0.0016	1.0000	0.0016	[0,1]		r_5: (BareNuclei < 2.500) ˄ (CellSize < 3.500) ˄ (Mitoses < 2.500) ˄ (NormalNucleoli < 8.500) → [1]
0.3616	0.9640	0.3485	[214,8]		r_6: ( ) → [0]

Combinations: 1446
Time: 0.7530136108398438

====== Classification performance of XGBoost ======
              precision    recall  f1-score   support

           0     0.9615    1.0000   